In [1]:
import pandas as pd
import numpy as np  

In [2]:
df = pd.read_csv('Trading.csv')
df = df.sort_values(by='Date', ascending=True)
print(df.head())

       Symbol        Date     Open     High      Low    Close Percent Change  \
1157  TRADING  2021-03-31  4368.87  4385.37  4092.36  4269.88         0.00 %   
1156  TRADING  2021-04-01  4236.88  4638.45  4203.89  4638.45         0.00 %   
1155  TRADING  2021-04-04  4638.45  4639.44  4368.87  4434.86         0.00 %   
1154  TRADING  2021-04-05  4351.05  4655.61  4351.05  4567.51         0.00 %   
1153  TRADING  2021-04-06  4189.37  4189.37  4135.26  4135.26         0.00 %   

     Volume Turn Over  
1157      -         -  
1156      -         -  
1155      -         -  
1154      -         -  
1153      -         -  


In [3]:
df['tp'] = (df['Close']+df['High']+df['Low'])/3

In [4]:
# Clean thousands separators before numeric conversion
df['Volume'] = pd.to_numeric(df['Volume'].astype(str).str.replace(',', '', regex=False), errors='coerce')
df['rmf'] = df['tp'] * df['Volume']

In [5]:
#Find postive and negative money flow
df['tp_diff'] = df['tp'].diff()
df['Postive_tp']= df['tp_diff'] > 0
df['Negative_tp']= df['tp_diff'] < 0


In [6]:
df['pos_flow'] = df['rmf'].where(df['Postive_tp'], 0)
df['neg_flow'] = df['rmf'].where(df['Negative_tp'], 0)

pos_sum = df['pos_flow'].rolling(14).sum()
neg_sum = df['neg_flow'].rolling(14).sum()

mfr = pos_sum / neg_sum
df['mfi'] = 100 - (100 / (1 + mfr))

In [7]:
df.tail()

,Symbol,Date,Open,High,Low,Close,Percent Change,Volume,Turn Over,tp,rmf,tp_diff,Postive_tp,Negative_tp,pos_flow,neg_flow,mfi
4,TRADING,2026-03-24,3788.15,4136.67,3788.14,4133.82,9.13 %,49228396.5,-,4019.543333,1.978757e+11,207.693333,True,False,1.978757e+11,0.000000e+00,77.862817
3,TRADING,2026-03-25,4063.10,4348.83,3991.67,4276.68,3.46 %,60402645.0,-,4205.726667,2.540370e+11,186.183333,True,False,2.540370e+11,0.000000e+00,79.450876
2,TRADING,2026-03-26,4276.69,4276.69,4064.53,4155.10,-2.84 %,9545076.3,-,4165.440000,3.975944e+10,-40.286667,False,True,0.000000e+00,3.975944e+10,73.434522
1,TRADING,2026-03-29,4155.11,4155.11,3957.38,4051.96,-2.48 %,34053438.6,-,4054.816667,1.380805e+11,-110.623333,False,True,0.000000e+00,1.380805e+11,64.940182
0,TRADING,2026-03-30,4051.96,4051.96,3898.81,3948.10,-2.56 %,19731837.9,-,3966.290000,7.826219e+10,-88.526667,False,True,0.000000e+00,7.826219e+10,61.409142


In [8]:
# Prepare data for plotting: convert date to datetime
df_plot = df.copy()
df_plot['Date'] = pd.to_datetime(df_plot['Date'])

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots with 2 rows and shared x-axis
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart", "Money Flow Index (MFI)")
)

# Convert date to string to use as categorical axis (preserves gaps)
df_plot['Date_str'] = df_plot['Date'].astype(str)

# Add candlestick chart to the first subplot
fig.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI line chart to the second subplot
fig.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2),
        fill='tozeroy',
        fillcolor='rgba(0, 0, 255, 0.2)'
    ),
    row=2, col=1
)

# Add horizontal line at 70 and 30 for overbought/oversold
fig.add_hline(y=70, line_dash="dash", line_color="red", annotation_text="Overbought", row=2)
fig.add_hline(y=30, line_dash="dash", line_color="green", annotation_text="Oversold", row=2)

# Update layout
fig.update_layout(
    height=700,
    title_text="Trading View with MFI Indicator",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white'
)

# Update y-axes labels
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="MFI", row=2, col=1)

fig.show()

In [10]:
# Swing detection function
def find_swings(series, lookback=10):
    highs, lows = [], []
    for i in range(lookback, len(series) - lookback):
        window = series.iloc[i-lookback: lookback + i +1]
        current = float(series.iloc[i])
        if current == float(window.max()):
            highs.append(i)
        if current == float(window.min()):
            lows.append(i)
    return highs, lows

# Find swing highs and lows for price
high_idx, low_idx = find_swings(df_plot["Close"])

# Divergence detection
regular_bullish, regular_bearish = [], []
hidden_bullish, hidden_bearish = [], []

# Regular Bullish: Price LL, MFI HL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i - 1], low_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        regular_bullish.append((p1, p2))

# Regular Bearish: Price HH, MFI LH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i - 1], high_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        regular_bearish.append((p1, p2))

# Hidden Bullish: Price HL, MFI LL
for i in range(1, len(low_idx)):
    p1, p2 = low_idx[i-1], low_idx[i]
    if df_plot["Close"].iloc[p2] > df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] < df_plot["mfi"].iloc[p1]:
        hidden_bullish.append((p1, p2))

# Hidden Bearish: Price LH, MFI HH
for i in range(1, len(high_idx)):
    p1, p2 = high_idx[i-1], high_idx[i]
    if df_plot["Close"].iloc[p2] < df_plot["Close"].iloc[p1] and df_plot["mfi"].iloc[p2] > df_plot["mfi"].iloc[p1]:
        hidden_bearish.append((p1, p2))

print("Regular Bullish Divergences:", regular_bullish)
print("Regular Bearish Divergences:", regular_bearish)
print("Hidden Bullish Divergences:", hidden_bullish)
print("Hidden Bearish Divergences:", hidden_bearish)

Regular Bullish Divergences: [(1099, 1112)]
Regular Bearish Divergences: [(303, 328), (387, 404), (469, 485), (534, 553), (553, 589), (751, 799)]
Hidden Bullish Divergences: [(294, 379), (394, 417), (635, 665), (856, 889)]
Hidden Bearish Divergences: [(140, 190), (190, 303), (361, 387), (589, 645), (659, 689), (689, 717), (961, 1016), (1037, 1081)]


In [11]:
# Plot divergence lines on the chart
fig_div = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    row_heights=[0.7, 0.3],
    subplot_titles=("Candlestick Chart with Divergences", "Money Flow Index (MFI)")
)

# Add candlestick chart
fig_div.add_trace(
    go.Candlestick(
        x=df_plot['Date_str'],
        open=df_plot['Open'],
        high=df_plot['High'],
        low=df_plot['Low'],
        close=df_plot['Close'],
        name='OHLC',
        increasing_line_color='green',
        decreasing_line_color='red',
    ),
    row=1, col=1
)

# Add MFI
fig_div.add_trace(
    go.Scatter(
        x=df_plot['Date_str'],
        y=df_plot['mfi'],
        mode='lines',
        name='MFI',
        line=dict(color='blue', width=2)
      
    ),
    row=2, col=1
)

# Regular Bullish on PRICE
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bearish on PRICE
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bullish on PRICE
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='lime', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=1, col=1
    )

# Hidden Bearish on PRICE
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['Close'].iloc[p1], df_plot['Close'].iloc[p2]],
            mode='lines+markers',
            line=dict(color='orange', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=1, col=1
    )

# Regular Bullish on MFI
for p1, p2 in regular_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='green', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Regular Bearish on MFI
for p1, p2 in regular_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False,
            hovertemplate='Regular Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bullish on MFI
for p1, p2 in hidden_bullish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='lime', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bullish<extra></extra>'
        ),
        row=2, col=1
    )

# Hidden Bearish on MFI
for p1, p2 in hidden_bearish:
    fig_div.add_trace(
        go.Scatter(
            x=[df_plot['Date_str'].iloc[p1], df_plot['Date_str'].iloc[p2]],
            y=[df_plot['mfi'].iloc[p1], df_plot['mfi'].iloc[p2]],
            mode='lines',
            line=dict(color='orange', dash='dot', width=2),
            showlegend=False,
            hovertemplate='Hidden Bearish<extra></extra>'
        ),
        row=2, col=1
    )

# Add overbought/oversold lines (faded)
fig_div.add_hline(y=70, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)
fig_div.add_hline(y=30, line_dash="dash", line_color="rgba(128, 128, 128, 0.2)", row=2)

# Add legend entries manually
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='green', dash='dash', width=2), 
                         name='Regular Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='red', dash='dash', width=2), 
                         name='Regular Bearish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='lime', dash='dot', width=2), 
                         name='Hidden Bullish'), row=1, col=1)
fig_div.add_trace(go.Scatter(x=[None], y=[None], mode='lines', 
                         line=dict(color='orange', dash='dot', width=2), 
                         name='Hidden Bearish'), row=1, col=1)

# Update layout
fig_div.update_layout(
    height=700,
    width=1400,
    title_text="Trading View with MFI Divergences",
    xaxis_rangeslider_visible=False,
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0, y=1)
)

fig_div.update_yaxes(title_text="Price", row=1, col=1)
fig_div.update_yaxes(title_text="MFI", row=2, col=1)

# Print summary
print(f"Regular Bullish Divergences: {len(regular_bullish)}")
print(f"Regular Bearish Divergences: {len(regular_bearish)}")
print(f"Hidden Bullish Divergences: {len(hidden_bullish)}")
print(f"Hidden Bearish Divergences: {len(hidden_bearish)}")

fig_div.show()

Regular Bullish Divergences: 1
Regular Bearish Divergences: 6
Hidden Bullish Divergences: 4
Hidden Bearish Divergences: 8


In [12]:
y_r_bullish = [y for _, y in regular_bullish]
y_r_bearish = [y for _, y in regular_bearish]
y_h_bullish = [y for _, y in hidden_bullish]
y_h_bearish = [y for _, y in hidden_bearish]

In [13]:
candle_intervals= [5,10,20,40,60]
results=[]

In [14]:
all_data=[]
for index in y_r_bullish:
    all_data.append((index, "Regular Bullish"))
for index in y_r_bearish:
    all_data.append((index, "Regular Bearish"))
for index in y_h_bullish:
    all_data.append((index, "Hidden Bullish"))
for index in y_h_bearish:
    all_data.append((index, "Hidden Bearish"))
    

In [15]:
symbol= df["Symbol"][0]
symbol

'TRADING'

In [16]:
for i,pattern in all_data:
    if i+max(candle_intervals)>= len(df):
        continue
    for j in candle_intervals:
        price_now = df["Close"].iloc[i]
        price_future = df["Close"].iloc[i+j]

        pc= (price_future-price_now)/price_now *100

        results.append({
            "Sector": symbol,
            "Pattern":pattern,
            "Interval":j,
            "Start Index":i,
            "EndIndex": i+j,
            "Price Now":price_now,
            "Price After Interval":price_future,
            "Percentage Change": pc
        })
        

In [17]:
results = pd.DataFrame(results)

In [18]:
results

,Sector,Pattern,Interval,Start Index,EndIndex,Price Now,Price After Interval,Percentage Change
0,TRADING,Regular Bearish,5,328,333,2359.43,2141.71,-9.227652
1,TRADING,Regular Bearish,10,328,338,2359.43,2114.33,-10.388102
2,TRADING,Regular Bearish,20,328,348,2359.43,1987.91,-15.746176
3,TRADING,Regular Bearish,40,328,368,2359.43,1778.22,-24.633492
4,TRADING,Regular Bearish,60,328,388,2359.43,1861.44,-21.106369
...,...,...,...,...,...,...,...,...
85,TRADING,Hidden Bearish,5,1081,1086,3859.95,3804.92,-1.425666
86,TRADING,Hidden Bearish,10,1081,1091,3859.95,3793.24,-1.728261
87,TRADING,Hidden Bearish,20,1081,1101,3859.95,3773.09,-2.250288
88,TRADING,Hidden Bearish,40,1081,1121,3859.95,3784.92,-1.943808


In [19]:
results.to_excel(f"{symbol}_MFI_Divergence.xlsx",index=False)